# 教程：创建混合检索管道

- **难度级别**：中级
- **完成时间**：15分钟
- **使用的组件**：[`DocumentSplitter`](https://docs.haystack.deepset.ai/docs/documentsplitter)、[`SentenceTransformersDocumentEmbedder`](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder)、[`DocumentJoiner`](https://docs.haystack.deepset.ai/docs/documentjoiner)、[`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore)、[`InMemoryBM25Retriever`](https://docs.haystack.deepset.ai/docs/inmemorybm25retriever)、[`InMemoryEmbeddingRetriever`](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever) 以及 [`TransformersSimilarityRanker`](https://docs.haystack.deepset.ai/docs/transformerssimilarityranker)
- **先决条件**：无
- **目标**：完成本教程后，你将了解如何创建混合检索以及它在何时有用。

## 概述

**混合检索**结合了基于关键词的检索技术和基于嵌入的检索技术，充分利用了两种方法的优势。本质上，密集嵌入擅长捕捉查询的上下文细微差别，而基于关键词的方法则擅长匹配关键词。

在很多情况下，像BM25这样简单的基于关键词的方法比密集检索表现更好（例如在医疗保健等特定领域），因为密集模型需要在数据上进行训练。有关混合检索的更多详细信息，请查看[博客文章：混合文档检索](https://haystack.deepset.ai/blog/hybrid-retrieval)。

## 安装Haystack

使用`pip`安装Haystack和其他所需的包：

In [ ]:
%%bash

pip install haystack-ai
pip install "datasets>=2.6.1"
pip install "sentence-transformers>=4.1.0"
pip install accelerate

## 初始化文档存储库

你将通过初始化一个文档存储库（DocumentStore）来开始创建自己的问答系统。文档存储库用于存储你的系统在回答问题时所依据的文档。在本教程中，你将使用[`InMemoryDocumentStore`](https://docs.haystack.deepset.ai/docs/inmemorydocumentstore)。

In [ ]:
from haystack.document_stores.in_memory import InMemoryDocumentStore

document_store = InMemoryDocumentStore()

> `InMemoryDocumentStore`是最易于上手的DocumentStore。它不需要外部依赖，是小型项目和调试的理想选择。但它在处理较大的文档集合时扩展性不佳，因此不适合生产系统。要了解Haystack支持的不同类型的外部数据库，请参阅[DocumentStore集成](https://haystack.deepset.ai/integrations?type=Document+Store&version=2.0)。

## 获取和处理文档

作为文档，你将使用PubMed摘要。Hugging Face Hub上有很多来自PubMed的数据集；在本教程中，你将使用[anakin87/medrag-pubmed-chunk](https://huggingface.co/datasets/anakin87/medrag-pubmed-chunk)。

然后，你将通过一个简单的for循环从该数据集创建文档。
PubMed数据集中的每个数据点都有4个特征：
* *pmid*
* *title*（标题）
* *content*（内容）：摘要
* *contents*（全部内容）：摘要+标题

在搜索时，你将使用*contents*特征。其他特征将作为元数据存储，你可以用它们来**美观地打印**搜索结果，或者用于[元数据过滤](https://docs.haystack.deepset.ai/docs/metadata-filtering)。

In [ ]:
from datasets import load_dataset
from haystack import Document

dataset = load_dataset("anakin87/medrag-pubmed-chunk", split="train")

docs = []
for doc in dataset:
    docs.append(
        Document(content=doc["contents"], meta={"title": doc["title"], "abstract": doc["content"], "pmid": doc["id"]})
    )

In [ ]:
import json

# 查看前两个文档的字典结构
for i in range(2):
    doc_dict = docs[i].to_dict()
    # 使用 json.dumps 让输出更美观 (indent=2)
    print(json.dumps(doc_dict, indent=2, ensure_ascii=False))

## 使用管道对文档进行索引

创建一个管道，将数据及其嵌入存储在文档存储中。对于这个管道，你需要一个[DocumentSplitter](https://docs.haystack.deepset.ai/docs/documentsplitter)来将文档分割成512个单词的块，一个[SentenceTransformersDocumentEmbedder](https://docs.haystack.deepset.ai/docs/sentencetransformersdocumentembedder)来创建用于密集检索的文档嵌入，以及一个[DocumentWriter](https://docs.haystack.deepset.ai/docs/documentwriter)来将文档写入文档存储。

作为嵌入模型，你将使用Hugging Face上的[BAAI/bge-small-en-v1.5](https://huggingface.co/BAAI/bge-small-en-v1.5)。你也可以随意测试Hugging Face上的其他模型，或者使用另一个[Embedder](https://docs.haystack.deepset.ai/docs/embedders)来切换模型提供商。

> 如果这一步对你来说耗时太长，可以将嵌入模型替换为更小的模型，例如`sentence-transformers/all-MiniLM-L6-v2`或`sentence-transformers/all-mpnet-base-v2`。确保根据你的模型的令牌限制更新`split_length`。

In [ ]:
from haystack.components.writers import DocumentWriter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.preprocessors.document_splitter import DocumentSplitter
from haystack import Pipeline
from haystack.utils import ComponentDevice

document_splitter = DocumentSplitter(split_by="word", split_length=512, split_overlap=32)
document_embedder = SentenceTransformersDocumentEmbedder(
    model="BAAI/bge-small-en-v1.5", device=ComponentDevice.from_str("cuda:0")
)
document_writer = DocumentWriter(document_store)

indexing_pipeline = Pipeline()
indexing_pipeline.add_component("document_splitter", document_splitter)
indexing_pipeline.add_component("document_embedder", document_embedder)
indexing_pipeline.add_component("document_writer", document_writer)

indexing_pipeline.connect("document_splitter", "document_embedder")
indexing_pipeline.connect("document_embedder", "document_writer")

indexing_pipeline.run({"document_splitter": {"documents": docs}})

文档及其嵌入被存储在`InMemoryDocumentStore`中，现在是创建混合检索管道的时候了✅

## 创建混合检索管道

混合检索指的是结合多种检索方法以提升整体性能。在搜索系统的语境中，混合检索管道会同时执行传统的基于关键词的搜索和密集向量搜索，之后使用交叉编码器模型对结果进行排序。这种组合使搜索系统能够利用不同方法的优势，提供更准确且更多样化的结果。

以下是混合检索管道所需的步骤：

### 1) 初始化检索器和嵌入器

初始化一个[InMemoryEmbeddingRetriever](https://docs.haystack.deepset.ai/docs/inmemoryembeddingretriever)和[InMemoryBM25Retriever](https://docs.haystack.deepset.ai/docs/inmemorybm25retriever)，以执行密集检索和基于关键词的检索。对于密集检索，你还需要一个[SentenceTransformersTextEmbedder](https://docs.haystack.deepset.ai/docs/sentencetransformerstextembedder)，它通过使用与索引管道中相同的嵌入模型`BAAI/bge-small-en-v1.5`来计算搜索查询的嵌入：

In [ ]:
from haystack.components.retrievers.in_memory import InMemoryBM25Retriever, InMemoryEmbeddingRetriever
from haystack.components.embedders import SentenceTransformersTextEmbedder

text_embedder = SentenceTransformersTextEmbedder(
    model="BAAI/bge-small-en-v1.5", device=ComponentDevice.from_str("cuda:0")
)
embedding_retriever = InMemoryEmbeddingRetriever(document_store)
bm25_retriever = InMemoryBM25Retriever(document_store)

### 2) 合并检索结果

Haystack在[`DocumentJoiner`](https://docs.haystack.deepset.ai/docs/documentjoiner)中提供了多种合并方法，可用于不同的使用场景，例如`merge`（合并）和`reciprocal_rank_fusion`（ reciprocal rank fusion， reciprocal rank fusion是一种将多个排序结果融合的算法，此处保留原名）。在本示例中，你将使用默认的`concatenate`（拼接）模式来合并来自两个检索器的文档，因为[Ranker](https://docs.haystack.deepset.ai/docs/rankers)将是对文档相关性进行排序的主要组件。

In [ ]:
from haystack.components.joiners import DocumentJoiner

document_joiner = DocumentJoiner()

### 3) 对结果进行排序

使用[TransformersSimilarityRanker](https://docs.haystack.deepset.ai/docs/transformerssimilarityranker)，它通过交叉编码器模型对所有检索到的文档与给定搜索查询的相关性进行评分。在本示例中，你将使用[BAAI/bge-reranker-base](https://huggingface.co/BAAI/bge-reranker-base)模型对检索到的文档进行排序，但你也可以用Hugging Face上的其他交叉编码器模型替换该模型。

In [ ]:
from haystack.components.rankers import TransformersSimilarityRanker

ranker = TransformersSimilarityRanker(model="BAAI/bge-reranker-base")

### 4) 创建混合检索管道

将所有初始化的组件添加到你的管道中并进行连接。

In [ ]:
from haystack import Pipeline

hybrid_retrieval = Pipeline()
hybrid_retrieval.add_component("text_embedder", text_embedder)
hybrid_retrieval.add_component("embedding_retriever", embedding_retriever)
hybrid_retrieval.add_component("bm25_retriever", bm25_retriever)
hybrid_retrieval.add_component("document_joiner", document_joiner)
hybrid_retrieval.add_component("ranker", ranker)

hybrid_retrieval.connect("text_embedder", "embedding_retriever")
hybrid_retrieval.connect("bm25_retriever", "document_joiner")
hybrid_retrieval.connect("embedding_retriever", "document_joiner")
hybrid_retrieval.connect("document_joiner", "ranker")

### 5) 可视化管道（可选）

要了解如何构建混合检索管道，请使用管道的[draw()](https://docs.haystack.deepset.ai/docs/drawing-pipeline-graphs)方法。如果您在Google Colab上运行此笔记本，生成的文件将保存在侧边栏的“文件”部分。

In [ ]:
# hybrid_retrieval.draw("hybrid-retrieval.png")

## 测试混合检索

将查询传递给`text_embedder`、`bm25_retriever`和`ranker`，并运行检索管道：


In [ ]:
query = "apnea in infants"

result = hybrid_retrieval.run(
    {"text_embedder": {"text": query}, "bm25_retriever": {"query": query}, "ranker": {"query": query}}
)

### 美观地打印结果
创建一个函数来打印一种“搜索页面”。

In [ ]:
def pretty_print_results(prediction):
    for doc in prediction["documents"]:
        print(doc.meta["title"], "\t", doc.score)
        print(doc.meta["abstract"])
        print("\n", "\n")

In [ ]:
pretty_print_results(result["ranker"])

## What's next

🎉 Congratulations! You've create a hybrid retrieval pipeline!

If you'd like to use this retrieval method in a RAG pipeline, check out [Tutorial: Creating Your First QA Pipeline with Retrieval-Augmentation](https://haystack.deepset.ai/tutorials/27_first_rag_pipeline) to learn about the next steps.

To stay up to date on the latest Haystack developments, you can [sign up for our newsletter](https://landing.deepset.ai/haystack-community-updates) or [join Haystack discord community](https://discord.gg/haystack).

Thanks for reading!